In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from src.forecasting import (
    BACKTEST_METRIC_COLUMNS,
    EVAL_WINDOW_24H,
    EVAL_WINDOW_48H,
    load_consumption_series,
    mean_absolute_percentage_error,
    per_horizon_percentage_error,
    plot_forecast_comparison,
    plot_horizon_errors,
    run_chronos_backtest,
    run_chronos_eval,
)

## Données

Série horaire continue, index UTC, produite par `01_data_preparation.ipynb`.

In [ ]:
y_series = load_consumption_series("../data/processed/energy_consumption_realized_resampled.csv")
print(f"{len(y_series)} heures, {y_series.index.min()} -> {y_series.index.max()}")
y_series.describe()

## Backtest rolling origin (grille modèle × fenêtre)

Pour chaque couple (modèle, fenêtre) : `NUM_WINDOWS` holdouts, chacun décalé de `STRIDE_HOURS` vers le passé. Les contextes sont prédits en un seul appel batché, le coût est donc proche d'une prévision unique.

Lecture : `mape_chronos` vs `mape_baseline` en moyenne (précision) et en écart-type (stabilité). Le MAE est en MW, l'unité dans laquelle raisonne le métier.

In [ ]:
MODELS = ("amazon/chronos-t5-tiny", "amazon/chronos-t5-base")
WINDOWS = (EVAL_WINDOW_24H, EVAL_WINDOW_48H)
NUM_WINDOWS = 10  # 10 origines différentes, une par jour
STRIDE_HOURS = 24
NUM_SAMPLES = 20  # suffisant pour la médiane ; monter à 100 pour des quantiles fins

runs = []
for model_id in MODELS:
    for spec in WINDOWS:
        print(f"{model_id} × {spec.name}")
        results = run_chronos_backtest(
            y_series,
            spec,
            model_id=model_id,
            num_windows=NUM_WINDOWS,
            stride_hours=STRIDE_HOURS,
            num_samples=NUM_SAMPLES,
        )
        results["model"] = model_id.removeprefix("amazon/")
        results["window"] = spec.name
        runs.append(results)

backtest = pd.concat(runs, ignore_index=True)
summary = (
    backtest.groupby(["model", "window"])[list(BACKTEST_METRIC_COLUMNS)]
    .agg(["mean", "std"])
    .round(2)
)
summary

In [ ]:
# Régularité : part des fenêtres où Chronos bat la baseline (1.0 = toujours, 0.5 = pile ou face).
backtest["chronos_wins"] = backtest["mape_chronos"] < backtest["mape_baseline"]
backtest.groupby(["model", "window"])["chronos_wins"].mean().rename("win_rate").round(2)

## Inspection visuelle (dernière fenêtre uniquement)

Qualitatif : forme de la courbe, largeur de la bande P10–P90, erreur par pas d'horizon. Le chiffre qui compte reste le backtest ci-dessus, pas le MAPE de cette seule fenêtre.

In [ ]:
result = run_chronos_eval(
    y_series, EVAL_WINDOW_24H, model_id="amazon/chronos-t5-tiny", num_samples=100
)
for msg in result.sanity_messages:
    print(f"Note : {msg}")

plot_forecast_comparison(
    result.ground_truth,
    result.baseline,
    result.median,
    result.low,
    result.high,
    horizon=result.horizon,
    title=f"{result.model_id} — {result.window_name} (dernière fenêtre)",
)

mape_chronos = mean_absolute_percentage_error(result.ground_truth, result.median)
mape_baseline = mean_absolute_percentage_error(result.ground_truth, result.baseline)
print(f"MAPE Chronos  : {mape_chronos:.2f}%")
print(f"MAPE baseline : {mape_baseline:.2f}%")

err_pct = per_horizon_percentage_error(result.ground_truth, result.median)
plot_horizon_errors(err_pct, horizon=result.horizon)